# Point 4 — RBM Hyperparameter Study

We compare RBMs with different hyperparameters on MNIST digits **0 / 1 / 2** and use
CD-based diagnostics to assess their quality (assignment, Point 4):

- **Sec. 1** — Contrastive Divergence steps `Nt`;
- **Sec. 2** — L1 regularisation `γ`;
- **Sec. 3** — optimiser and learning rate;
- **Sec. 4** — `spins` encoding;
- **Sec. 5** — `potts` encoding;
- **Sec. 6** — energy diagnostics and inter-class barriers.

> Cells must be executed **in order**. Models are saved to disk: on subsequent runs,
> sections with *smart reload* will load them without retraining.


## 0. Setup and imports


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.datasets import fetch_openml

from rbm_pro_max import RBM, load_results
from rbm_plots import (
    plot_metrics, plot_cd_analysis, plot_filters,
    plot_weight_evolution, plot_convergence_comparison,
)

plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 16
})


## 1. Data loading, preprocessing and shared configuration

Same dataset and preprocessing as Point 1 (digits 0/1/2, pixels in `[0,1]` + Gaussian noise).
`MODEL_BASE` and `TRAIN_BASE` collect the shared hyperparameters; each section copies them
and changes **one** parameter at a time.


In [ ]:
X_original, Y_original = fetch_openml("mnist_784", version=1,
                                      return_X_y=True, as_frame=False)
X_original = np.array(X_original, dtype=np.float32)
Y_original = np.array(Y_original, dtype=int)

mask = (Y_original == 0) | (Y_original == 1) | (Y_original == 2)
X_filtered = X_original[mask]
Y_filtered = Y_original[mask]
print(f"Filtrati 0,1,2 : {X_filtered.shape}")

In [ ]:
# --- Preprocessing ---
X_train = X_filtered / 255.0
X_train = X_train + np.random.normal(0, 0.01, X_train.shape)
X_train = np.clip(X_train, 0, 1)

D      = X_train.shape[1]    # visible units (784)
side   = 28                  # image side length
L_BASE = 12                  # reference number of hidden units
base_glorot_scale = np.sqrt(4.0 / (D + L_BASE))

print(f"Range [{X_train.min():.3f}, {X_train.max():.3f}]  mean {X_train.mean():.3f}  D={D}")

# --- Shared configuration ---
MODEL_BASE = dict(
    optimizer="rmsprop",
    lr=0.05,
    lr_schedule="constant",
    gamma=0.001,             # L1 regularisation
    weight_decay=0.0,        # L2 disabled
    potts=False,
    spins=False,
)

TRAIN_BASE = dict(
    Nepoch=150, Nmini=20, N_ini=10, N_fin=500,
    Nt=2,                    # Contrastive Divergence steps
    persistent=False,
    hinton_init=True,
    history_every=5, metrics_every=5,
    n_metrics_samples=300,
    verbose=False,
)


## 2. Helper functions

Helpers for metric comparison, filter visualisation, and energy diagnostics
(per-class free energy, inter-class barriers).


In [ ]:
# ----- METRICS & FILTERS ---
def load_runs(root, keys):
    """Load {label: results} from root/key folders on disk."""
    return {str(k): load_results(Path(root) / str(k)) for k in keys}

def compare_metric_plot(runs, title=""):
    """Three panels: reconstruction error, free-energy gap, pseudo-likelihood."""
    metrics = [
        ("reconstruction_error", "Reconstruction error"),
        ("free_energy_gap",      "Free-energy gap"),
        ("pseudo_likelihood",    "Pseudo-likelihood"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
    cmap = plt.colormaps["tab10"]
    for ax, (key, ylabel) in zip(axes, metrics):
        for i, (label, res) in enumerate(runs.items()):
            m = res["metrics"]
            if len(m["epochs"]) == 0:
                continue
            ax.plot(m["epochs"], m[key], label=label, color=cmap(i % 10), lw=1.5)
        ax.set(xlabel="Epoch", ylabel=ylabel, title=ylabel)
        ax.legend(fontsize=12); ax.grid(alpha=0.3)
    if title:
        fig.suptitle(title, fontsize=13)
    return fig

def filters_comparison(runs, side, selected=None, ncols=6):
    """Show filter grids for selected model variants."""
    keys = selected or list(runs.keys())
    for k in keys:
        w_dict = runs[k]["weights"]
        W_matrix = w_dict.get("W", w_dict.get("w"))
        plot_filters(W_matrix, side=side, title=f"Filters - {k}", ncols=ncols)

# ----- ENERGY & BARRIERS ---
def per_class_free_energy(model, data, labels):
    """Mean free energy of the model for each class."""
    labels = np.asarray(labels, dtype=int)
    return {int(c): float(model.free_energy(data[labels == c]))
            for c in np.unique(labels)}

def plot_class_fe(fe_dict, title="Free energy per class"):
    classes = sorted(fe_dict.keys())
    fig, ax = plt.subplots(figsize=(7, 3.5), constrained_layout=True)
    ax.bar(classes, [fe_dict[c] for c in classes],
           color="#5E3C99", alpha=0.8, width=0.6)
    ax.set(xlabel="Digit class", ylabel="Mean free energy F(v)",
           title=title, xticks=classes)
    ax.grid(axis="y", alpha=0.3)
    return fig

def energy_barrier_plot(model, prototypes, class_a, class_b, n_steps=80):
    """Free-energy profile along the linear interpolation between two prototypes."""
    x0, x1 = prototypes[class_a], prototypes[class_b]
    alphas = np.linspace(0, 1, n_steps)
    X_int  = np.stack([(1 - a) * x0 + a * x1 for a in alphas])
    FE = np.array([model.free_energy(X_int[i:i+1]) for i in range(n_steps)])
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    ax.plot(alphas, FE, color="#884488", lw=1.8)
    ax.axvline(0, color="#2266CC", lw=1.2, ls="--", label=f"Class {class_a}")
    ax.axvline(1, color="#CC4422", lw=1.2, ls="--", label=f"Class {class_b}")
    ax.fill_between(alphas, FE.min(), FE, alpha=0.1, color="#884488")
    ax.set(xlabel=f"alpha (0 -> {class_a}, 1 -> {class_b})",
           ylabel="Free energy F(v)",
           title=f"Energy barrier: {class_a} <-> {class_b}")
    ax.legend(); ax.grid(alpha=0.3)
    return fig

def barrier_matrix_plot(model, prototypes, n_steps=60):
    """Heatmap of barrier heights for all pairs of classes."""
    classes = sorted(prototypes.keys())
    n = len(classes)
    mat = np.zeros((n, n))
    for i, ca in enumerate(classes):
        for j, cb in enumerate(classes):
            if i == j:
                continue
            x0, x1 = prototypes[ca], prototypes[cb]
            X_int = np.stack([(1 - a) * x0 + a * x1
                              for a in np.linspace(0, 1, n_steps)])
            FE = np.array([model.free_energy(X_int[k:k+1]) for k in range(n_steps)])
            mat[i, j] = FE.max() - (FE[0] + FE[-1]) / 2
    fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
    im = ax.imshow(mat, cmap="viridis")
    ax.set(xticks=range(n), yticks=range(n),
           xticklabels=classes, yticklabels=classes,
           xlabel="Target class", ylabel="Source class",
           title="Energy barrier matrix (Delta F)")
    plt.colorbar(im, ax=ax, label="Barrier height (Delta F)")
    return fig

def load_model_pro_max(res, n_v, n_h):
    """Reconstruct an RBM object from saved weights/biases."""
    clean_cfg = {k: v for k, v in MODEL_BASE.items()
                 if k not in ('n_v', 'n_h', 'D', 'L')}
    m = RBM(n_v=n_v, n_h=n_h, seed=42, w_init_scale=0.1, **clean_cfg)
    w = res["weights"]
    m.W      = w.get("W", w.get("w"))
    m.v_bias = w.get("v_bias", w.get("a"))
    m.h_bias = w.get("h_bias", w.get("b"))
    return m

print("Helpers ready.")


## 3. Section 0 — Reference base model

Base model with `L = 12`, used as reference and for energy diagnostics (Sec. 6).


In [ ]:
print("Training reference base model ...")

pro_base_model = RBM(
    n_v=D, n_h=L_BASE, seed=12345,
    w_init_scale=base_glorot_scale, **MODEL_BASE,
)
pro_base_res = pro_base_model.train(X_train, save_dir="base_run_pro_max", **TRAIN_BASE)
print("Done. Diagnostics:")

fig_base_metrics = plot_metrics(pro_base_res)
fig_base_cd      = plot_cd_analysis(pro_base_res)

W_matrix = pro_base_res["weights"].get("W", pro_base_res["weights"].get("w"))


In [ ]:
fig_base_filters = plot_filters(W_matrix, side=side,
                                title=f"Base model - L={L_BASE}")
fig_base_wevol   = plot_weight_evolution(pro_base_res, side=side)

fig_base_metrics.suptitle("Base model - metrics", fontsize=13)
fig_base_cd.suptitle("Base model - CD analysis", fontsize=13)
plt.show()


## 4. Section 1 — Contrastive Divergence steps `Nt`

Comparison for `Nt ∈ {1, 2, 5, 10, 20}`. Hinton (Sec. 14): when weights are small the
chain mixes quickly, so CD-1 is already a good approximation; increasing *n* brings CD
closer to maximum likelihood but makes the gradient noisier.


In [ ]:
Nt_list = [1, 2, 5, 10, 20]
Nt_runs = {}

print("Studying the effect of CD steps (Nt) ...")
for nt in Nt_list:
    print(f" -> training with Nt = {nt} ...")
    model_nt = RBM(n_v=D, n_h=L_BASE, seed=12345,
                   w_init_scale=base_glorot_scale, **MODEL_BASE)
    cfg = TRAIN_BASE.copy()
    cfg["Nt"] = nt
    Nt_runs[str(nt)] = model_nt.train(X_train, save_dir=f"Nt_run/{nt}", **cfg)

print("Training complete. Comparison plots:")


In [ ]:
fig_Nt_compare = compare_metric_plot(Nt_runs, title="Effect of CD steps (Nt)")

In [ ]:
fig_Nt_grad    = plot_convergence_comparison(Nt_runs, metric="gradient_rms",
                                             title="Gradient RMS - Nt comparison")
plt.show()

In [ ]:
figs_Nt_filters = filters_comparison(Nt_runs, side, selected=["1", "5", "20"])

## 5. Section 2 — L1 regularisation `γ`

Comparison for `γ ∈ {0, 1e-4, 1e-3, 1e-2, 1e-1}`. L1 penalty drives many weights to
exactly zero, allowing only a few to grow: this yields more localised and interpretable
*receptive fields* (Hinton, Sec. 10).


In [ ]:
gamma_list = [0.0, 1e-4, 1e-3, 1e-2, 1e-1]
gamma_runs = {}

print("Studying the effect of L1 regularisation (gamma) ...")
for g in gamma_list:
    print(f" -> training with gamma = {g} ...")
    cfg_model = MODEL_BASE.copy()
    cfg_model["gamma"] = g
    model_gamma = RBM(n_v=D, n_h=L_BASE, seed=12345,
                      w_init_scale=base_glorot_scale, **cfg_model)
    gamma_runs[str(g)] = model_gamma.train(X_train, save_dir=f"gamma_run/{g}",
                                           **TRAIN_BASE)

print("Training complete. Comparison plots:")


In [ ]:
fig_gamma_compare = compare_metric_plot(gamma_runs,
                                        title="Effect of L1 regularization (gamma)")

In [ ]:
fig_gamma_grad = plot_convergence_comparison(gamma_runs, metric="gradient_rms",
                                                title="Gradient RMS - gamma comparison")
plt.show()

In [ ]:
figs_gamma_filters = filters_comparison(gamma_runs, side,
                                        selected=["0.0", "0.001", "0.1"])

## 6. Section 3 — Optimiser and learning rate

Comparison of RMSprop vs SGD at three learning rates. *Smart reload*: if folder
`opt_run/<label>` already contains `metrics.npz`, the model is reloaded instead of retrained.


In [ ]:
opt_variants = {
    "RMSprop_lr005": dict(optimizer="rmsprop", lr=0.05, lr_schedule="constant"),
    "RMSprop_lr001": dict(optimizer="rmsprop", lr=0.01, lr_schedule="constant"),
    "RMSprop_lr010": dict(optimizer="rmsprop", lr=0.10, lr_schedule="constant"),
    "SGD_lr001":     dict(optimizer="sgd",     lr=0.01, lr_schedule="constant"),
    "SGD_lr005":     dict(optimizer="sgd",     lr=0.05, lr_schedule="constant"),
    "SGD_lr010":     dict(optimizer="sgd",     lr=0.10, lr_schedule="constant"),
}
opt_runs = {}

print("Studying optimiser and learning rate ...")
for label, kw in opt_variants.items():
    folder = f"opt_run/{label}"
    if os.path.exists(os.path.join(folder, "metrics.npz")):
        print(f" -> [DISK] {label}: loading ...")
        opt_runs[label] = load_results(folder)
    else:
        print(f" -> [TRAIN] {label} ...")
        cfg_model = MODEL_BASE.copy()
        cfg_model.update(kw)
        model_opt = RBM(n_v=D, n_h=L_BASE, seed=12345,
                        w_init_scale=base_glorot_scale, **cfg_model)
        opt_runs[label] = model_opt.train(X_train, save_dir=folder, **TRAIN_BASE)

print("Ready. Comparison plots:")


In [ ]:
fig_opt_compare = compare_metric_plot(opt_runs,
                                      title="Optimizer / learning-rate comparison")
fig_opt_grad    = plot_convergence_comparison(opt_runs, metric="gradient_rms",
                                              title="Gradient RMS - optimizer comparison")
figs_opt_filters = filters_comparison(opt_runs, side,
                                      selected=["RMSprop_lr005", "SGD_lr005"])
plt.show()

## 7. Section 4 — `spins` encoding

With `spins=True` both hidden and visible units use a **±1** encoding
(Ising model: `tanh` activation, sampling in {−1, +1}).

Data must be mapped from [0, 1] to [−1, +1] before training: `X_spins = 2·X_train − 1`.

Hinton initialisation of visible biases uses `arctanh(mean)` instead of `logit(mean)`,
consistent with the ±1 encoding. `free_energy`, `pseudo_likelihood` and AIS also use the
correct formula `log(2·cosh(x))` for ±1 units.


In [ ]:
spins_flags = [False, True]
spins_runs = {}

print("Studying spins encoding ...")
for flag in spins_flags:
    folder = f"spins_run/{flag}"
    if os.path.exists(os.path.join(folder, "metrics.npz")):
        print(f" -> [DISK] spins={flag}: loading ...")
        spins_runs[str(flag)] = load_results(folder)
    else:
        print(f" -> [TRAIN] spins={flag} ...")
        cfg_model = MODEL_BASE.copy()
        cfg_model["spins"] = flag
        model_spins = RBM(n_v=D, n_h=L_BASE, seed=12345,
                          w_init_scale=base_glorot_scale, **cfg_model)
        # Map data to [-1, +1] when using spins encoding
        X_input = 2.0 * X_train - 1.0 if flag else X_train
        spins_runs[str(flag)] = model_spins.train(X_input, save_dir=folder,
                                                  **TRAIN_BASE)

print("Ready. Comparison plots:")


In [ ]:
fig_spins_compare  = compare_metric_plot(spins_runs,
                                         title="Spins encoding: False vs True")
figs_spins_filters = filters_comparison(spins_runs, side, selected=["False", "True"])
plt.show()

## 8. Section 5 — `potts` encoding

### Theoretical background

The **Potts** model is the multi-state generalisation of the Ising model. With `potts=True`
the entire hidden layer becomes **a single softmax group**: exactly **one** hidden unit
is active at a time (mutual exclusion). This acts as a strong structural regulariser but
severely limits model capacity — with `L = 12` the hidden layer can encode at most
~log₂(12) ≈ 3.6 bits.

$$P(z_k = 1 \mid v) = \frac{\exp(b_k + \sum_i v_i W_{ik})}{\sum_m \exp(b_m + \sum_i v_i W_{im})}$$

**Expected behaviour:** initially slower convergence of reconstruction error (the softmax
constraint is harder than sigmoid), and typically sharper, more specialised filters.

> **Note:** as with `spins`, `free_energy` / `pseudo_likelihood` / AIS use the Bernoulli
> formula and are not exact for softmax units. The `sparsity` metric is also degenerate
> for Potts (always equals 1/L).


In [ ]:
potts_flags = [False, True]
potts_runs = {}

print("Studying potts encoding ...")
for flag in potts_flags:
    folder = f"potts_run/{flag}"
    if os.path.exists(os.path.join(folder, "metrics.npz")):
        print(f" -> [DISK] potts={flag}: loading ...")
        potts_runs[str(flag)] = load_results(folder)
    else:
        print(f" -> [TRAIN] potts={flag} ...")
        cfg_model = MODEL_BASE.copy()
        cfg_model["potts"] = flag
        model_potts = RBM(n_v=D, n_h=L_BASE, seed=12345,
                          w_init_scale=base_glorot_scale, **cfg_model)
        potts_runs[str(flag)] = model_potts.train(X_train, save_dir=folder,
                                                  **TRAIN_BASE)

print("Ready. Comparison plots:")


In [ ]:
fig_potts_compare  = compare_metric_plot(potts_runs,
                                         title="Potts encoding: False vs True")
figs_potts_filters = filters_comparison(potts_runs, side, selected=["False", "True"])
plt.show()

## 9. Section 6 — Energy diagnostics and inter-class barriers

Mean free energy per class on the base model and **energy barriers** ΔF along the linear
interpolation between class prototypes. High intermediate barriers explain the
**quasi-ergodicity** of CD observed in Point 1 (connection with Point 3).


In [ ]:
print("Energy analysis of the base model ...")

# Class prototypes (per-class mean image)
prototypes = {int(c): X_train[Y_filtered == c].mean(axis=0)
              for c in np.unique(Y_filtered)}

# Per-class free energy on the base model
fe_by_class  = per_class_free_energy(pro_base_model, X_train, Y_filtered)
fig_fe_class = plot_class_fe(fe_by_class,
                             title=f"Free energy per class - base model (L={L_BASE})")

# Energy barrier matrix for all digit pairs
print("Computing energy barrier matrix ...")
fig_barrier_mat = barrier_matrix_plot(pro_base_model, prototypes, n_steps=60)

# Barrier profiles for pairs 0<->1, 1<->2, 0<->2
fig_b01 = energy_barrier_plot(pro_base_model, prototypes, 0, 1)
fig_b12 = energy_barrier_plot(pro_base_model, prototypes, 1, 2)
fig_b02 = energy_barrier_plot(pro_base_model, prototypes, 0, 2)
plt.show()


In [ ]:
# Per-class free energy comparison across Nt variants
print("Comparing per-class free energy across Nt variants ...")

fig_fe_Nt, axes = plt.subplots(1, len(Nt_list),
                               figsize=(4 * len(Nt_list), 3.5),
                               constrained_layout=True)
if len(Nt_list) == 1:
    axes = [axes]

for ax, nt in zip(axes, Nt_list):
    m_nt   = load_model_pro_max(Nt_runs[str(nt)], n_v=D, n_h=L_BASE)
    fe_nt  = per_class_free_energy(m_nt, X_train, Y_filtered)
    cls_nt = sorted(fe_nt.keys())
    ax.bar(cls_nt, [fe_nt[c] for c in cls_nt],
           color="#2266CC", alpha=0.7, width=0.6)
    ax.set(title=f"Nt = {nt}", xlabel="Class", ylabel="Mean F(v)")
    ax.grid(axis="y", alpha=0.3)

fig_fe_Nt.suptitle("Per-class free energy vs Nt", fontsize=13)
plt.show()
print("Energy diagnostics complete.")
